# 01 — Exploración inicial del corpus
**Autor:** Giuliano Crenna, Juan Ignacio Pace (UGR)
**Fecha:** 2026-09-03
**Descripción:** Conteos por fuente/clase, longitudes, % vacíos.
**Stage:** Etapa 3 — EDA previa al modelado.
## Parámetros
Los siguientes parámetros pueden sobreescribirse con `papermill`:
- `DATA_DIR`: ruta al directorio `data/` (default `./data`).
- `SEED`: semilla (default 42).
- `OUT_DIR`: dónde guardar figuras y tablas (default `reports`).


In [0]:
# %% [code]
import os
from pathlib import Path
import pandas as pd
import yaml
import matplotlib.pyplot as plt
DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
SEED = int(os.environ.get("SEED", 42))
OUT_DIR = Path(os.environ.get("OUT_DIR", "reports"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Fix de semilla.
import random
random.seed(SEED)
import numpy as np
np.random.seed(SEED)


In [0]:
# %% [code]
# Cargar el corpus procesado (o interim si todavía no se corrió make data).
processed = DATA_DIR / "processed" / "corpus_v1.parquet"
if processed.exists():
    df = pd.read_parquet(processed)
    print(f"corpus procesado: {len(df)} filas, {df['user_id'].nunique()} usuarios únicos")
else:
    print(f"WARN: no existe {processed}. Construyendo desde interim/*.parquet")
    frames = []
    for p in (DATA_DIR / "interim").glob("*/data.parquet"):
        frames.append(pd.read_parquet(p))
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    print(f"interim: {len(df)} filas")
df.head()


In [0]:
# %% [code]
# Conteos por fuente y por clase.
print("=== Conteos por fuente ===")
print(df["source"].value_counts())
print()
print("=== Conteos por label ===")
print(df["label"].value_counts().sort_index())
print()
print("=== Conteos por fuente × label ===")
print(df.groupby(["source", "label"]).size().unstack(fill_value=0))


In [0]:
# %% [code]
# Longitudes de texto (caracteres y tokens).
df["len_chars"] = df["text_clean"].fillna("").str.len()
df["len_tokens"] = df["text_clean"].fillna("").str.split().str.len()
print(df[["len_chars", "len_tokens"]].describe())
print()
print("% vacíos por fuente:")
print((df["text_clean"].fillna("").str.len() == 0).groupby(df["source"]).mean() * 100)


In [0]:
# %% [code]
# Histograma de longitudes por fuente.
fig, ax = plt.subplots(figsize=(10, 5))
for src, sub in df.groupby("source"):
    ax.hist(sub["len_tokens"].clip(upper=200), bins=50, alpha=0.5, label=src)
ax.set_xlabel("# tokens (clip a 200)")
ax.set_ylabel("frecuencia")
ax.set_title("Distribución de longitudes por fuente")
ax.legend()
plt.tight_layout()
out = OUT_DIR / "figures" / "eda_01_longitudes_por_fuente.png"
out.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out, dpi=120)
plt.show()
print(f"figura → {out}")


## Conclusiones
- Documentar acá los hallazgos (luego de ejecutar).
- Si alguna fuente tiene 0% de mensajes no-vacíos, hay que revisar
  `make_dataset.py` para esa fuente.
